# edge-medvqa: full run on Colab

Runtime > Change runtime type > **T4 GPU**. Total time is roughly 30-40 minutes.

1. Zero-shot baseline
2. LoRA fine-tuning (2 epochs on 1,793 pairs)
3. Fine-tuned model on GPU
4. Merged + int8 model on CPU (the edge version)

In [ ]:
!git clone -q https://github.com/zubairzulfiqar/edge-medvqa
%cd edge-medvqa
!pip -q install -e ".[dev]"
!pytest -q tests/test_metrics.py

## 1. Zero-shot baseline (no training)

In [ ]:
!python -m medvqa.evaluate --label 'zero-shot (GPU)' --out runs/results.json

## 2. LoRA fine-tuning

In [ ]:
!python -m medvqa.train --epochs 2 --batch-size 8 --lr 2e-4 --out runs/lora

## 3. Fine-tuned, on GPU

In [ ]:
!python -m medvqa.evaluate --adapter runs/lora/adapter --label 'LoRA fine-tuned (GPU)' --out runs/results.json

## 4. Edge version: LoRA merged, int8, CPU

Colab CPUs are slow and shared, so latency here is an upper bound; a recent laptop is usually faster.

In [ ]:
!python -m medvqa.evaluate --adapter runs/lora/adapter --int8 --device cpu --label 'LoRA merged + int8 (CPU)' --out runs/results.json
!python -m medvqa.evaluate --adapter runs/lora/adapter --device cpu --dtype float32 --limit 100 --label 'LoRA merged fp32 (CPU, 100 q)' --out runs/results.json

## 5. Results table and loss curve

Paste the table into the README.

In [ ]:
import json
from medvqa.evaluate import to_markdown
print(to_markdown(json.load(open('runs/results.json'))))

import matplotlib.pyplot as plt
loss = [h['loss'] for h in json.load(open('runs/lora/loss.json'))]
plt.figure(figsize=(6,3)); plt.plot(loss, lw=0.8); plt.xlabel('batch'); plt.ylabel('loss'); plt.title('LoRA training loss'); plt.tight_layout(); plt.savefig('runs/loss.png', dpi=120); plt.show()

## 6. Keep the adapter (optional)

The LoRA adapter is only a few MB. Save it to Drive, or push it to the Hugging Face Hub with `model.push_to_hub`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/edge-medvqa && cp -r runs /content/drive/MyDrive/edge-medvqa/